# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [6]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
import pandas as pd
df = pd.read_csv("orders.csv")
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [18]:
df["amount"] = pd.to_numeric(
    df["amount"].replace(" ", ""),
    errors="coerce"
)
print("После преобразования amount:", len(df))
df["city"] = df["city"].str.strip().str.lower()
df["city"] = df["city"].replace({
    "almaty": "алматы",
    "astana": "астана",
    "shymkent": "шымкент"
})

print(df["city"].value_counts())

После преобразования amount: 5283
city
алматы       2119
астана       1339
шымкент       776
караганда     519
актобе        387
атырау        143
Name: count, dtype: int64


In [20]:
df = df.drop_duplicates()

print("После удаления дубликатов:", len(df))

После удаления дубликатов: 5283


In [22]:
df.shape

(5283, 8)

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [25]:
# 2
paid = df[df["status"] == "оплачен"]

In [35]:
df["order_date"] < "2026-10"
sep = paid[
    (paid["order_date"] >= "2026-09") &
    (paid["order_date"] < "2026-10")
]["amount"].sum()
print("Сентябрь:", sep)
oct = paid[
    (paid["order_date"] >= "2026-10") &
    (paid["order_date"] < "2026-11")
]["amount"].sum()
print("Октябрь:", oct)
change = (oct - sep) / sep * 100
print("Изменение:", round(change, 2), "%")

Сентябрь: 55243390.0
Октябрь: 44419510.0
Изменение: -19.59 %


In [36]:
for city in df["city"].unique():

    city_data = paid[paid["city"] == city]

    sep = city_data[
        (city_data["order_date"] >= "2026-09") &
        (city_data["order_date"] < "2026-10")
    ]["amount"].sum()

    oct = city_data[
        (city_data["order_date"] >= "2026-10") &
        (city_data["order_date"] < "2026-11")
    ]["amount"].sum()

    change = (oct - sep) / sep * 100

    print(city, round(change, 2), "%")

алматы -28.57 %
астана -2.03 %
караганда -7.59 %
шымкент -11.29 %
атырау 4.96 %
актобе -1.42 %


## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

# 3
## Журнал очистки данных

| Шаг очистки | Удалено строк | Падение выручки |
|---|---:|---:|
| Исходные данные | 0 | -22.7% |
| Удаление дубликатов | 180 | -19.6% |
| Удаление TEST | 3 | -2.3% |

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [38]:
# 4
df["delivery_days"].isna().sum()

np.int64(158)

In [46]:
df[df["delivery_days"].isna()]["city"].value_counts()

city
алматы       63
астана       40
караганда    24
шымкент      22
атырау        5
актобе        4
Name: count, dtype: int64

In [47]:
sh = df[df["city"] == "шымкент"].copy()
print("Количество заказов:", len(sh))

Количество заказов: 776


In [44]:
a = sh["delivery_days"].dropna()

print("Количество строк:", len(a))
print("Доставка за 2 дня:", (a <= 2).mean() * 100)

Количество строк: 754
Доставка за 2 дня: 29.84084880636605


In [45]:
b = sh["delivery_days"].fillna(
    sh["delivery_days"].median()
)

print("Количество строк:", len(b))
print("Доставка за 2 дня:", (b <= 2).mean() * 100)

Количество строк: 776
Доставка за 2 дня: 28.994845360824744


## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [49]:
# 5
df = pd.read_csv("orders.csv")

# Ошибка 1: сохраняем результат удаления дубликатов
df = df.drop_duplicates()

# Ошибка 2: очищаем названия городов
df["city"] = df["city"].str.strip().str.lower()

# Заменяем латинские названия
df["city"] = df["city"].replace({
    "almaty": "алматы",
    "astana": "астана",
    "shymkent": "шымкент"
})

print(df["city"].nunique(), len(df))

6 5283


## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [52]:
# 6
print(df["order_date"].dtype)

datetime64[ns]


In [53]:
df["order_date"] = pd.to_datetime(df["order_date"])

In [55]:
days = df["order_date"].dt.day_name()

print(days.value_counts())

order_date
Thursday     786
Tuesday      785
Friday       782
Saturday     775
Wednesday    774
Sunday       697
Monday       684
Name: count, dtype: int64


In [56]:
counts = days.value_counts()

print("Самый загруженный день:", counts.idxmax())
print("Меньше всего заказов:", counts.idxmin())

Самый загруженный день: Thursday
Меньше всего заказов: Monday


In [57]:
max_orders = counts.max()
min_orders = counts.min()
difference = (max_orders - min_orders) / max_orders * 100
print("Разница:", round(difference, 2), "%")

Разница: 12.98 %


## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [ ]:
# 7

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [ ]:
# 8